# Notebook 11 — Final Dataset Preprocessing & Validation
**Project:** Adaptive Road Vision  
**Dataset:** 15,000 images from Notebook 10 (`final_condition_balanced_v2`)  
**Task:** YOLO object detection

This notebook prepares and validates the final dataset for detector training. It checks the 70/20/10 split, image and label paths, YOLO bounding boxes, synthetic-Fog restrictions, class distribution, and visual examples. It writes a verified dataset configuration and audit reports.

**Important:** This is an object-detection dataset. Do **not** resize/augment images independently with a classification transform: bounding boxes must receive the same geometric transform. Ultralytics/YOLO should handle training-time resizing and detection-safe augmentation. This notebook does not modify images or labels.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import json, random, warnings
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
                  Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate Adaptive_Road_Vision. Open this notebook from the project "
        "folder/notebooks, or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "datasets" / "final_condition_balanced_v2"
METADATA_CSV = DATASET_ROOT / "metadata" / "final_dataset_metadata.csv"
DATASET_YAML = DATASET_ROOT / "dataset.yaml"
REPORT_DIR = PROJECT_ROOT / "results" / "preprocessing"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TOTAL = 15000
EXPECTED_SPLITS = {"train": 10500, "test": 3000, "val": 1500}
CLASS_NAMES = [
    "person", "rider", "car", "truck", "bus",
    "train", "motorcycle", "bicycle", "traffic light", "traffic sign"
]
NUM_CLASSES = len(CLASS_NAMES)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Metadata:", METADATA_CSV)
print("Expected images:", EXPECTED_TOTAL)

## 2. Confirm the final dataset structure

In [ ]:
required_dirs = [
    DATASET_ROOT / "images" / "train", DATASET_ROOT / "images" / "val",
    DATASET_ROOT / "images" / "test", DATASET_ROOT / "labels" / "train",
    DATASET_ROOT / "labels" / "val", DATASET_ROOT / "labels" / "test",
]
missing_dirs = [str(p) for p in required_dirs if not p.is_dir()]
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Final dataset folder not found: {DATASET_ROOT}")
if missing_dirs:
    raise FileNotFoundError("Missing expected YOLO directories:\n" + "\\n".join(missing_dirs))
if not METADATA_CSV.is_file():
    raise FileNotFoundError(f"Expected Notebook 10 metadata was not found: {METADATA_CSV}")
if not DATASET_YAML.is_file():
    print("WARNING: dataset.yaml is missing; a verified one will be written in Section 9.")

metadata = pd.read_csv(METADATA_CSV)
print("Metadata rows/columns:", metadata.shape)
print("Columns:", metadata.columns.tolist())
display(metadata.head())

## 3. Standardize metadata and resolve paths

In [ ]:
required_columns = {"split", "image_path", "label_path", "condition", "synthetic", "source_split"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(
        f"Metadata is missing required columns: {sorted(missing_columns)}. "
        "Use the metadata produced by Notebook 10."
    )

df = metadata.copy()
df["split"] = df["split"].astype(str).str.strip().str.lower().replace(
    {"validation": "val", "valid": "val", "training": "train", "testing": "test"}
)
df["condition"] = df["condition"].fillna("Unknown").astype(str).str.strip()
df["synthetic"] = df["synthetic"].map(
    lambda x: str(x).strip().lower() in {"true", "1", "yes", "y"}
)
df["source_split"] = df["source_split"].astype(str).str.strip().str.lower().replace(
    {"validation": "val", "valid": "val", "training": "train", "testing": "test"}
)

def resolve_path(value):
    p = Path(str(value))
    if p.is_absolute():
        return p
    options = [PROJECT_ROOT / p, DATASET_ROOT / p]
    for candidate in options:
        if candidate.exists():
            return candidate
    return options[0]

df["image_path"] = df["image_path"].map(resolve_path)
df["label_path"] = df["label_path"].map(resolve_path)

if not set(df["split"]).issubset({"train", "val", "test"}):
    raise ValueError(f"Unexpected split names: {sorted(set(df['split']))}")
if df.empty:
    raise ValueError("Metadata contains zero records.")

display(df["split"].value_counts().reindex(["train", "val", "test"], fill_value=0).rename("images").to_frame())
display(pd.crosstab(df["split"], df["condition"], margins=True))

## 4. Verify the 15,000-image target and split counts

In [ ]:
actual_splits = df["split"].value_counts().to_dict()
assert len(df) == EXPECTED_TOTAL, f"Expected {EXPECTED_TOTAL} rows, found {len(df)}."
assert actual_splits == EXPECTED_SPLITS, (
    f"Split mismatch. Expected {EXPECTED_SPLITS}; got {actual_splits}"
)
assert df["image_path"].astype(str).is_unique, "Duplicate image paths detected."
assert df["label_path"].astype(str).is_unique, "Duplicate label paths detected."

print("PASS: exactly 15,000 metadata records")
print("PASS: train=10,500, val=1,500, test=3,000")
print("PASS: image and label paths are unique")

## 5. Validate image and label files

In [ ]:
missing_images = df.loc[~df["image_path"].map(lambda p: p.is_file())]
missing_labels = df.loc[~df["label_path"].map(lambda p: p.is_file())]
file_issues = []
for _, row in missing_images.iterrows():
    file_issues.append({"split": row["split"], "path": str(row["image_path"]), "issue": "missing image"})
for _, row in missing_labels.iterrows():
    file_issues.append({"split": row["split"], "path": str(row["label_path"]), "issue": "missing label"})

if file_issues:
    pd.DataFrame(file_issues).to_csv(REPORT_DIR / "preprocessing_file_issues.csv", index=False)
    raise FileNotFoundError(
        f"Found {len(file_issues)} missing image/label files. "
        f"See {REPORT_DIR / 'preprocessing_file_issues.csv'}"
    )

print("PASS: all images and labels exist")

## 6. Validate image readability and YOLO labels

In [ ]:
label_issues = []
empty_labels = []
image_sizes = []

for row in df.itertuples(index=False):
    image_path = Path(row.image_path)
    label_path = Path(row.label_path)

    try:
        with Image.open(image_path) as im:
            im.verify()
        with Image.open(image_path) as im:
            width, height = im.size
            if width <= 0 or height <= 0:
                label_issues.append({"path": str(image_path), "issue": "invalid image dimensions"})
            image_sizes.append({"split": row.split, "width": width, "height": height})
    except Exception as exc:
        label_issues.append({"path": str(image_path), "issue": f"unreadable image: {exc}"})
        continue

    try:
        lines = label_path.read_text(encoding="utf-8").splitlines()
    except Exception as exc:
        label_issues.append({"path": str(label_path), "issue": f"cannot read label: {exc}"})
        continue

    if not lines:
        empty_labels.append({
            "image_name": getattr(row, "image_name", image_path.name),
            "split": row.split, "condition": row.condition,
            "synthetic": row.synthetic, "image_path": str(image_path),
            "label_path": str(label_path)
        })
        continue

    for line_no, line in enumerate(lines, start=1):
        values = line.split()
        if len(values) != 5:
            label_issues.append({"path": str(label_path), "issue": f"line {line_no}: expected 5 values"})
            continue
        try:
            class_float = float(values[0])
            coords = np.asarray([float(v) for v in values[1:]], dtype=np.float64)
            if not np.isfinite(class_float) or not class_float.is_integer():
                raise ValueError("class id must be an integer")
            class_id = int(class_float)
            x, y, w, h = coords
            if class_id < 0 or class_id >= NUM_CLASSES:
                label_issues.append({"path": str(label_path), "issue": f"line {line_no}: class id {class_id} outside 0..{NUM_CLASSES-1}"})
            if not np.isfinite(coords).all():
                label_issues.append({"path": str(label_path), "issue": f"line {line_no}: non-finite coordinates"})
            elif not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                label_issues.append({"path": str(label_path), "issue": f"line {line_no}: invalid normalized xywh"})
            elif x - w/2 < -1e-6 or y - h/2 < -1e-6 or x + w/2 > 1+1e-6 or y + h/2 > 1+1e-6:
                label_issues.append({"path": str(label_path), "issue": f"line {line_no}: bounding box extends outside image"})
        except Exception as exc:
            label_issues.append({"path": str(label_path), "issue": f"line {line_no}: invalid numeric values ({exc})"})

pd.DataFrame(image_sizes).to_csv(REPORT_DIR / "image_dimensions.csv", index=False)
pd.DataFrame(empty_labels, columns=[
    "image_name", "split", "condition", "synthetic", "image_path", "label_path"
]).to_csv(REPORT_DIR / "empty_label_files_review.csv", index=False)

if label_issues:
    pd.DataFrame(label_issues).to_csv(REPORT_DIR / "preprocessing_integrity_issues.csv", index=False)
    display(pd.DataFrame(label_issues).head(25))
    raise ValueError(f"Found {len(label_issues)} image/label integrity issues. See preprocessing_integrity_issues.csv")

print("PASS: all images readable and all non-empty YOLO labels valid")
print("Empty labels (review to confirm these are intentional background images):", len(empty_labels))

## 7. Check synthetic Fog policy and split leakage

In [ ]:
synthetic_df = df[df["synthetic"]].copy()
original_df = df[~df["synthetic"]].copy()

if not synthetic_df.empty:
    assert synthetic_df["split"].eq("train").all(), "Synthetic images found outside train."
    assert synthetic_df["source_split"].eq("train").all(), "Synthetic images must originate from training data."
    assert synthetic_df["condition"].str.lower().eq("fog").all(), "Synthetic records should be marked Fog."

# Original source IDs must not cross splits. The Notebook 10 metadata uses source_id.
if "source_id" in original_df.columns:
    split_ids = {
        split: set(original_df.loc[original_df["split"].eq(split), "source_id"].astype(str))
        for split in ["train", "val", "test"]
    }
    assert not split_ids["train"] & split_ids["val"], "Original train/val source leakage detected."
    assert not split_ids["train"] & split_ids["test"], "Original train/test source leakage detected."
    assert not split_ids["val"] & split_ids["test"], "Original val/test source leakage detected."

if {"source_split", "split"}.issubset(original_df.columns):
    assert original_df["source_split"].eq(original_df["split"]).all(), "Original image crossed its source split."

print("Original images:", len(original_df))
print("Synthetic Fog images:", len(synthetic_df))
print("PASS: synthetic-only-train and source split checks")

## 8. Analyze condition and object-class distribution

In [ ]:
condition_table = pd.crosstab(df["split"], df["condition"]).reindex(
    index=["train", "val", "test"], fill_value=0
)
display(condition_table)
condition_table.to_csv(REPORT_DIR / "condition_distribution_by_split.csv")

class_counts = Counter()
class_image_counts = Counter()
for row in df.itertuples(index=False):
    seen = set()
    for line in Path(row.label_path).read_text(encoding="utf-8").splitlines():
        values = line.split()
        if len(values) == 5:
            cid = int(float(values[0]))
            class_counts[(row.split, cid)] += 1
            seen.add(cid)
    for cid in seen:
        class_image_counts[(row.split, cid)] += 1

class_rows = []
for cid, name in enumerate(CLASS_NAMES):
    rec = {"class_id": cid, "class_name": name}
    for split in ["train", "val", "test"]:
        rec[f"{split}_instances"] = class_counts[(split, cid)]
        rec[f"{split}_images"] = class_image_counts[(split, cid)]
    class_rows.append(rec)
class_table = pd.DataFrame(class_rows)
display(class_table)
class_table.to_csv(REPORT_DIR / "yolo_class_distribution.csv", index=False)

ax = condition_table.plot(kind="bar", figsize=(10, 5))
ax.set_title("Condition distribution by split")
ax.set_ylabel("Images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORT_DIR / "condition_distribution_by_split.png", dpi=160)
plt.show()

## 9. Verify or create `dataset.yaml`

In [ ]:
try:
    import yaml
except ImportError as exc:
    raise ImportError("Install PyYAML with: pip install pyyaml") from exc

yaml_config = {
    "path": str(DATASET_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)}
}
DATASET_YAML.write_text(
    yaml.safe_dump(yaml_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)
print(DATASET_YAML.read_text(encoding="utf-8"))
print("Saved verified YOLO configuration:", DATASET_YAML)

## 10. Visualize image and bounding-box examples

In [ ]:
def draw_yolo_boxes(ax, image_path, label_path, title=""):
    with Image.open(image_path) as im:
        im = im.convert("RGB")
        width, height = im.size
        ax.imshow(im)
    ax.set_title(title, fontsize=9)
    for line in Path(label_path).read_text(encoding="utf-8").splitlines():
        vals = line.split()
        if len(vals) != 5:
            continue
        cid = int(float(vals[0]))
        x, y, w, h = map(float, vals[1:])
        x1, y1 = (x - w/2) * width, (y - h/2) * height
        rect = patches.Rectangle((x1, y1), w*width, h*height,
                                 linewidth=1.5, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        label = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else str(cid)
        ax.text(x1, max(0, y1-3), label, color="black", fontsize=7,
                bbox=dict(facecolor="lime", alpha=0.75, pad=1, edgecolor="none"))
    ax.axis("off")

sample_df = df.sample(n=min(8, len(df)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, row in zip(axes.flatten(), sample_df.itertuples(index=False)):
    draw_yolo_boxes(ax, row.image_path, row.label_path,
                    f"{row.split} | {row.condition} | synthetic={row.synthetic}")
for ax in axes.flatten()[len(sample_df):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Save preprocessing summary and final acceptance checks

In [ ]:
summary = {
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "metadata_csv": str(METADATA_CSV),
    "dataset_yaml": str(DATASET_YAML),
    "task": "YOLO object detection",
    "expected_total": EXPECTED_TOTAL,
    "actual_total": int(len(df)),
    "expected_split_counts": EXPECTED_SPLITS,
    "actual_split_counts": {k: int(v) for k, v in df["split"].value_counts().items()},
    "condition_counts": {str(k): int(v) for k, v in df["condition"].value_counts().items()},
    "synthetic_count": int(df["synthetic"].sum()),
    "class_names": CLASS_NAMES,
    "checks": {
        "exact_15000_records": bool(len(df) == EXPECTED_TOTAL),
        "exact_70_20_10_split": bool(actual_splits == EXPECTED_SPLITS),
        "all_images_and_labels_exist": bool(not len(missing_images) and not len(missing_labels)),
        "valid_yolo_labels": bool(len(label_issues) == 0),
        "synthetic_fog_train_only": bool(synthetic_df.empty or synthetic_df["split"].eq("train").all()),
    },
    "notes": [
        "Images and labels were not modified.",
        "Use a YOLO detection trainer so resizing and augmentation remain synchronized with bounding boxes.",
        "Inspect empty-label files and CycleGAN Fog box alignment before final training."
    ]
}
summary_path = REPORT_DIR / "preprocessing_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, default=lambda o: bool(o) if isinstance(o, (np.bool_, np.bool)) else str(o)), encoding="utf-8")
pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in summary["checks"].items()]).to_csv(
    REPORT_DIR / "preprocessing_acceptance_checks.csv", index=False
)
display(pd.DataFrame([{"split": s, "expected": EXPECTED_SPLITS[s],
                       "actual": int((df["split"] == s).sum())}
                      for s in ["train", "val", "test"]]))
print(json.dumps(summary["checks"], indent=2, default=lambda o: bool(o) if isinstance(o, (np.bool_, np.bool)) else str(o)))
assert all(summary["checks"].values()), "One or more preprocessing acceptance checks failed."
print("PREPROCESSING VALIDATION PASSED")
print("Reports saved to:", REPORT_DIR)

## Notebook complete — next step: detector training

Use `datasets/final_condition_balanced_v2/dataset.yaml` with your YOLO detector training notebook. Keep the split unchanged and do not run random augmentation on validation/test. For CycleGAN Fog samples, visually inspect inherited bounding boxes because style transfer can alter object visibility or geometry.

This notebook is a validation/preparation stage, not a training run. It deliberately leaves the 15,000 image files and YOLO labels unchanged.